# Task 3 — PostgreSQL Database Storage

## Objective
The objective of this task is to design and implement a PostgreSQL relational database system capable of persistently storing cleaned fintech review data, sentiment analysis outputs, and thematic classifications. This simulates a real-world data engineering workflow involving schema design, ETL processes, database insertion, and integrity verification.

In [2]:
import pandas as pd

df = pd.read_csv('../../data/bank_reviews_cleaned.csv')

print(df.columns)
print(df.head())

Index(['review', 'rating', 'date', 'bank', 'source'], dtype='str')
                                    review  rating        date bank  \
0                         thanks for you 😘       5  2026-05-15  CBE   
1                                it's okay       4  2026-05-15  CBE   
2  It's not allowing me to transfer money.       2  2026-05-15  CBE   
3          IT'S NOT WORK ON HUAWEI DEVICES       4  2026-05-15  CBE   
4                                      wow       4  2026-05-14  CBE   

        source  
0  Google Play  
1  Google Play  
2  Google Play  
3  Google Play  
4  Google Play  


In [7]:
df2 = pd.read_csv('../../data/reviews_with_sentiment.csv')

print(df2.columns)
print(df2.head())

Index(['review', 'rating', 'date', 'bank', 'source', 'sentiment_label',
       'sentiment_score', 'review_id'],
      dtype='str')
                                    review  rating        date bank  \
0                         thanks for you 😘       5  2026-05-15  CBE   
1                                it's okay       4  2026-05-15  CBE   
2  It's not allowing me to transfer money.       2  2026-05-15  CBE   
3          IT'S NOT WORK ON HUAWEI DEVICES       4  2026-05-15  CBE   
4                                      wow       4  2026-05-14  CBE   

        source sentiment_label  sentiment_score  review_id  
0  Google Play        positive         0.999622          1  
1  Google Play        positive         0.999828          2  
2  Google Play        negative         0.996865          3  
3  Google Play        negative         0.999691          4  
4  Google Play        positive         0.999592          5  


In [6]:
df3 = pd.read_csv('../../data/final_thematic_analysis.csv')

print(df3.columns)
print(df3.head())

Index(['review_id', 'review_text', 'sentiment_label', 'sentiment_score',
       'identified_theme'],
      dtype='str')
   review_id                              review_text sentiment_label  \
0          1                         thanks for you 😘        positive   
1          2                                it's okay        positive   
2          3  It's not allowing me to transfer money.        negative   
3          4          IT'S NOT WORK ON HUAWEI DEVICES        negative   
4          5                                      wow        positive   

   sentiment_score         identified_theme  
0         0.999622                    Other  
1         0.999828                    Other  
2         0.996865  Transaction Performance  
3         0.999691                    Other  
4         0.999592                    Other  


## Database Schema Design

Two relational tables were created:

### Banks Table
Stores metadata about each fintech application.

Fields:
- bank_id (Primary Key)
- bank_name
- app_name

### Reviews Table
Stores cleaned customer review data and NLP analysis results.

Fields:
- review_id (Primary Key)
- bank_id (Foreign Key)
- review_text
- rating
- review_date
- sentiment_label
- sentiment_score
- identified_theme
- source

A foreign key relationship was established between the reviews and banks tables to maintain relational integrity.

In [13]:
import pandas as pd

# Load datasets
reviews_df = pd.read_csv('../../data/reviews_with_sentiment.csv')
themes_df = pd.read_csv('../../data/final_thematic_analysis.csv')

# Merge using review_id
final_df = reviews_df.merge(
    themes_df[['review_id', 'identified_theme']],
    on='review_id',
    how='left'
)

# Save final merged dataset
final_df.to_csv('../../data/final_database_dataset.csv', index=False)
print(final_df.head())
print(final_df.columns)

print("Final merged dataset saved successfully!")

                                    review  rating        date bank  \
0                         thanks for you 😘       5  2026-05-15  CBE   
1                                it's okay       4  2026-05-15  CBE   
2  It's not allowing me to transfer money.       2  2026-05-15  CBE   
3          IT'S NOT WORK ON HUAWEI DEVICES       4  2026-05-15  CBE   
4                                      wow       4  2026-05-14  CBE   

        source sentiment_label  sentiment_score  review_id  \
0  Google Play        positive         0.999622          1   
1  Google Play        positive         0.999828          2   
2  Google Play        negative         0.996865          3   
3  Google Play        negative         0.999691          4   
4  Google Play        positive         0.999592          5   

          identified_theme  
0                    Other  
1                    Other  
2  Transaction Performance  
3                    Other  
4                    Other  
Index(['review', 'rating',

In [11]:
print(final_df.columns)
print(final_df.shape)
print(final_df.isnull().sum())

Index(['review', 'rating', 'date', 'bank', 'source', 'sentiment_label',
       'sentiment_score', 'review_id', 'identified_theme'],
      dtype='str')
(1135, 9)
review              0
rating              0
date                0
bank                0
source              0
sentiment_label     0
sentiment_score     0
review_id           0
identified_theme    0
dtype: int64


In [14]:
import pandas as pd

df = pd.read_csv('../../data/final_database_dataset.csv')

df.head()

,review,rating,date,bank,source,sentiment_label,sentiment_score,review_id,identified_theme
0,thanks for you 😘,5,2026-05-15,CBE,Google Play,positive,0.999622,1,Other
1,it's okay,4,2026-05-15,CBE,Google Play,positive,0.999828,2,Other
2,It's not allowing me to transfer money.,2,2026-05-15,CBE,Google Play,negative,0.996865,3,Transaction Performance
3,IT'S NOT WORK ON HUAWEI DEVICES,4,2026-05-15,CBE,Google Play,negative,0.999691,4,Other
4,wow,4,2026-05-14,CBE,Google Play,positive,0.999592,5,Other


## Data Preparation

The final dataset used for insertion was created by merging:
- Cleaned review data
- Sentiment analysis results
- Thematic analysis outputs

The resulting dataset contained:
- Review text
- Ratings
- Dates
- Bank names
- Sentiment labels and scores
- Identified themes

This consolidated dataset simulates a real-world ETL (Extract, Transform, Load) workflow commonly used in financial analytics systems.

## Data Insertion Pipeline

Python scripts using psycopg2 were used to:
1. Establish a PostgreSQL database connection
2. Insert bank metadata into the banks table
3. Map bank names to foreign keys
4. Insert processed review records into the reviews table

The insertion pipeline successfully populated the database with over 1,000 processed review entries.

## Database Verification Queries

SQL verification queries were executed to:
- Count reviews per bank
- Compute average ratings
- Validate absence of critical null values

These checks confirmed that:
- Relational integrity was preserved
- Ratings and sentiment values were inserted correctly
- The database schema functioned as intended

In [ ]:
"""
SELECT b.bank_name, COUNT(r.review_id)
FROM reviews r
JOIN banks b
ON r.bank_id = b.bank_id
GROUP BY b.bank_name;
"""